# Lab 03-01 · Probability in EE Language

Companion to chapter [03-01](../../chapters/03-01-probability-in-ee-language.md). Handout: [README.md](README.md).

Each part simulates a process with random numbers and compares the measured frequencies with the chapter's formulas. Cells marked **🔮 Predict** ask you to write a prediction *before* running. The random seed is fixed, so your numbers will match the handout's.

In [1]:
import math, sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
sys.path.insert(0, str(Path.cwd().parents[3] / "tools"))
try:
    from plotstyle import use_style; use_style()
except ImportError:
    pass
rng = np.random.default_rng(2026)

def p_any(n, p):
    # 1 - (1 - p)^n, computed without cancellation
    return -math.expm1(n * math.log1p(-p))

## Part 1 — Rare events and the union bound

A packet has $n$ = 12,000 bits, each wrong with probability $p$. Simulating every bit would be slow, so draw the *number* of errors per packet directly (a binomial draw, the subject of 03-02) for 200,000 packets.

**🔮 Predict:** at $p = 10^{-5}$ and $p = 10^{-4}$, how close is the union bound $np$ to the measured packet error rate?

In [2]:
n, packets = 12_000, 200_000
for p in (1e-6, 1e-5, 1e-4):
    errors = rng.binomial(n, p, size=packets)
    measured = np.mean(errors > 0)
    print(f"p = {p:.0e}: measured {measured:.5f}   exact {p_any(n, p):.5f}   union bound {n * p:.5f}")

p = 1e-06: measured 0.01163   exact 0.01193   union bound 0.01200
p = 1e-05: measured 0.11203   exact 0.11308   union bound 0.12000
p = 1e-04: measured 0.69864   exact 0.69882   union bound 1.20000


The union bound is accurate while $np \ll 1$ and fails as $np$ approaches 1. Note also the literal formula's trap at very small $p$:

In [3]:
p = 1e-12
print("literal 1 - (1 - p)**n :", 1 - (1 - p) ** n)
print("expm1/log1p form       :", p_any(n, p))
print("union bound n*p        :", n * p)

literal 1 - (1 - p)**n : 1.199973442833624e-08
expm1/log1p form       : 1.1999999928006001e-08
union bound n*p        : 1.2e-08


## Part 2 — Birthday collisions

**🔮 Predict:** with random 16-bit IDs ($N$ = 65,536), how many devices make a collision more likely than not?

In [4]:
N = 2**16
def no_collision(k, N):
    out = 1.0
    for i in range(k):
        out *= 1 - i / N
    return out
k50 = next(k for k in range(2, 5000) if 1 - no_collision(k, N) >= 0.5)
print("exact 50% point:", k50, "   approximation sqrt(2 N ln 2) =", round(math.sqrt(2 * N * math.log(2)), 1))

trials = 4000
for k in (100, 200, 302, 400):
    hits = sum(len(set(rng.integers(0, N, size=k))) < k for _ in range(trials))
    print(f"k = {k}: simulated {hits / trials:.3f}, formula {1 - no_collision(k, N):.3f}")

exact 50% point: 302    approximation sqrt(2 N ln 2) = 301.4
k = 100: simulated 0.072, formula 0.073
k = 200: simulated 0.266, formula 0.262
k = 302: simulated 0.495, formula 0.501
k = 400: simulated 0.705, formula 0.705


## Part 3 — A test floor

Simulate a million dies through the test of Example 03-01.4: yield 90%, 99% of bad dies caught, 2% of good dies wrongly rejected.

**🔮 Predict:** how many bad dies ship per million shipped, and what fraction of the scrap is good?

In [5]:
dies = 1_000_000
good = rng.random(dies) < 0.90
u = rng.random(dies)
passes = np.where(good, u >= 0.02, u >= 0.99)        # good: fail w.p. 0.02; bad: pass w.p. 0.01
shipped_bad = np.sum(passes & ~good)
dppm = shipped_bad / np.sum(passes) * 1e6
good_in_scrap = np.sum(~passes & good) / np.sum(~passes)
print(f"pass fraction {np.mean(passes):.4f}  (formula 0.883)")
print(f"defect level {dppm:.0f} DPPM  (formula 1132.5)")
print(f"good dies among rejects {good_in_scrap:.4f}  (formula 0.1538)")

pass fraction 0.8832  (formula 0.883)
defect level 1173 DPPM  (formula 1132.5)
good dies among rejects 0.1538  (formula 0.1538)


## Part 4 — When independence fails: clustered defects

Dies come from wafer regions of two kinds: 80% of dies sit in clean regions, 20% in dirty ones where both failure modes are far more likely. *Within* a region the two failure modes are independent.

**🔮 Predict:** overall, will $P(L \cap M)$ be larger or smaller than $P(L)P(M)$?

In [6]:
dies = 1_000_000
dirty = rng.random(dies) < 0.2
pL = np.where(dirty, 0.15, 0.025)
pM = np.where(dirty, 0.12, 0.02)
L = rng.random(dies) < pL
M = rng.random(dies) < pM
print(f"P(L) = {L.mean():.4f}, P(M) = {M.mean():.4f}, P(L)P(M) = {L.mean() * M.mean():.5f}")
print(f"P(L and M) = {(L & M).mean():.5f}  -> ratio {(L & M).mean() / (L.mean() * M.mean()):.2f}")

P(L) = 0.0502, P(M) = 0.0400, P(L)P(M) = 0.00201
P(L and M) = 0.00396  -> ratio 1.98


The two failure modes are *conditionally* independent given the region, but dependent overall: a die that shows one failure is probably from a dirty region, so it is more likely to show the other. This is the structure of Example 03-01.2.

## Part 5 — Voting and common-mode failures

**🔮 Predict:** at module reliability $R = 0.9$, what does TMR give? And with $R = 0.4$?

In [7]:
trials = 400_000
for R in (0.9, 0.4):
    works = rng.random((trials, 3)) < R
    tmr = np.mean(works.sum(axis=1) >= 2)
    print(f"R = {R}: single {R:.3f}, TMR simulated {tmr:.4f}, formula {3 * R**2 - 2 * R**3:.4f}")

q, beta = 0.01, 0.1
cm = rng.random(trials) < beta * q                       # one shared event takes both down
ind = rng.random((trials, 2)) < (1 - beta) * q
both = np.mean(cm | ind.all(axis=1))
print(f"duplicated supplies: both fail {both:.5f}  (formula {beta * q + (1 - beta * q) * ((1 - beta) * q) ** 2:.5f}, independent {q * q:.5f})")

R = 0.9: single 0.900, TMR simulated 0.9718, formula 0.9720
R = 0.4: single 0.400, TMR simulated 0.3511, formula 0.3520
duplicated supplies: both fail 0.00104  (formula 0.00108, independent 0.00010)


## Part 6 — How long to prove a BER

A test sends $n = 3/p$ bits and passes if it sees no errors. If the true BER is exactly $p$, the number of errors in the test is Poisson with mean $np = 3$ (03-02).

**🔮 Predict:** what fraction of such tests would pass even though the BER is exactly at the limit?

In [8]:
tests = 200_000
errors = rng.poisson(3.0, size=tests)
print(f"fraction of tests passing at the limit: {np.mean(errors == 0):.4f}   (e^-3 = {math.exp(-3):.4f})")
for C in (0.90, 0.95, 0.99):
    print(f"confidence {C:.0%}: need {-math.log(1 - C):.3f}/p error-free bits")

fraction of tests passing at the limit: 0.0501   (e^-3 = 0.0498)
confidence 90%: need 2.303/p error-free bits
confidence 95%: need 2.996/p error-free bits
confidence 99%: need 4.605/p error-free bits


So a pass at $3/p$ bits leaves only a 5% chance that a link exactly at the BER limit would have passed: that is what "95% confidence" means here.

## Your turn

1. In Part 1, find the BER at which the union bound overstates the packet error rate by 10%.
2. In Part 3, raise the incoming yield to 98% and predict the new DPPM before running.
3. In Part 4, make the dirty regions 50% of dies. Does the dependence ratio grow or shrink? Why?
4. In Part 5, add a voter that fails with probability 0.001 and find the module reliability above which TMR stops helping (compare Problem 03-01.13).